# Cluster Non-Negative South Australia (SA) battery energy storage system (BESS) Households

## Purpose
This notebook explores the 44 selected Solar Analytics CICCADA BESS households that have no meaningful negative underlying demand in the processed selected sample. It does not write CSV, parquet, or other result files.

The notebook uses two simple clustering views:

1. PV size and operational battery-size proxy.
2. Typical weekly net-load profile after PV and battery.

## Inputs
- `SUMMARY_PATH`

## Run Flow
1. Purpose And Cohort Definition
2. Setup And Dependency Check
3. Load The 44 Non-Negative Households
4. Reconstruct PV And Battery Components
5. Method 1: PV And Battery Size Clustering
6. Method 1: Representative Households
7. Method 2: Typical Weekly Net-Load Profile Clustering
8. Method 2: Representative Households
9. Side-By-Side Cluster Interpretation
10. Notes And Caveats

## 1. Purpose And Cohort Definition

The next cell defines the expected cohort and clustering choices. The eligible households are selected sites with zero post-fill intervals below the current meaningful-negative threshold.

In [1]:
# Purpose: define cohort, signal, and clustering constants in one place.
# These values make the notebook deterministic and easy to audit.

EXPECTED_SIGNAL_PROFILE = "current_polarity_adjusted"
EXPECTED_NON_NEGATIVE_HOUSEHOLDS = 44
NEGATIVE_THRESHOLD_KW = -0.05
RANDOM_STATE = 42
N_CLUSTERS = 3
KMEANS_N_INIT = 50

TARGET_COLUMNS = [
    "underlying_load_kW",
    "net_load_with_pv_kW",
    "net_load_with_pv_and_battery_kW",
]

PROFILE_SIGNAL = "net_load_with_pv_and_battery_kW"
PROFILE_SIGNAL_LABEL = "Net load with PV and battery"

METHOD1_FEATURES = ["dc_capacity_kw", "battery_storage_abs_p95_kW"]
METHOD1_TARGET_LABELS = [
    "big PV + big battery",
    "small PV + big battery",
    "big PV + small battery",
]

DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
PROFILE_SLOTS_PER_DAY = 48
PROFILE_SLOTS_PER_WEEK = 7 * PROFILE_SLOTS_PER_DAY
PROFILE_TICK_VALUES = [day * PROFILE_SLOTS_PER_DAY + 24 for day in range(7)]
PROFILE_TICK_LABELS = DAYS

CLUSTER_COLORS = {
    "big PV + big battery": "#2f5597",
    "small PV + big battery": "#70ad47",
    "big PV + small battery": "#c55a11",
    "profile cluster 1": "#2f5597",
    "profile cluster 2": "#70ad47",
    "profile cluster 3": "#c55a11",
}

## 2. Setup And Dependency Check

This cell imports the analysis libraries, checks whether optional interactive plotting packages are available, and locates the cleaned SA BESS outputs used by the notebook. Plotly and ipywidgets are not required by the project, so the notebook includes fallback tables when they are missing.

In [2]:
# Purpose: import dependencies, discover paths, and report optional plotting support.
# The notebook uses sklearn for clustering and Plotly only for optional interactive figures.

import os
from pathlib import Path
import importlib.util
import itertools
import warnings

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
WIDGETS_AVAILABLE = importlib.util.find_spec("ipywidgets") is not None

if PLOTLY_AVAILABLE:
    import plotly.express as px
    import plotly.graph_objects as go
else:
    px = go = None


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until the PyNNLF repository root is found."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
def raw_data_root():
    """Locate the local source data tree, which is not distributed with this repository.

    Set the PYNNLF_RAW_DATA_DIR environment variable to the directory holding the
    "1. raw", "2. processed" and "3. cleaned" folders before running this notebook.

    Returns:
        Path: root of the local source data tree.
    """
    root = os.environ.get("PYNNLF_RAW_DATA_DIR")
    if not root:
        raise RuntimeError(
            "PYNNLF_RAW_DATA_DIR is not set. Point it at your local source data "
            "directory; see the Data section of the repository README."
        )
    return Path(root)


RAW_DATA_ROOT = raw_data_root()

CLEANED_SA_BESS_DIR = RAW_DATA_ROOT / "3. cleaned" / "SA BESS"
INTERMEDIATE_DIR = CLEANED_SA_BESS_DIR / "intermediate"
PROCESSED_DIR = CLEANED_SA_BESS_DIR / "processed"

SUMMARY_PATH = PROCESSED_DIR / "sa_bess_selected_household_summary.csv"
SITE_TIMESERIES_PATH = INTERMEDIATE_DIR / "sa_bess_selected_site_timeseries_5min.parquet"

for path in [SUMMARY_PATH, SITE_TIMESERIES_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected SA BESS input: {path}")

print(f"Repository root: {REPO_ROOT}")
print(f"Cleaned SA BESS folder: {CLEANED_SA_BESS_DIR}")
print(f"Plotly available: {PLOTLY_AVAILABLE}")
print(f"ipywidgets available: {WIDGETS_AVAILABLE}")

if not PLOTLY_AVAILABLE or not WIDGETS_AVAILABLE:
    display(Markdown(
        "**Interactive dependency note:** Plotly and ipywidgets are optional for this repository. "
        "To enable all interactive views, install them in the active notebook environment, "
        "for example `python -m pip install plotly ipywidgets`. This notebook does not modify dependency files."
    ))

Repository root: <local path redacted>
Cleaned SA BESS folder: <local path redacted>
Plotly available: True
ipywidgets available: True


## 3. Load The 44 Non-Negative Households

This section loads the selected-household summary, filters to households with no meaningful negative underlying demand, and reads only those 44 households from the selected-site 5-minute parquet.

In [3]:
# Purpose: load the non-negative household cohort and its 5-minute target series.
# Filtering the parquet by site_id keeps the notebook focused on the 44-household cohort.


def parse_mixed_datetime(series: pd.Series) -> pd.Series:
    """Parse the mixed date strings written by the SA BESS workflow."""
    return pd.to_datetime(series, errors="coerce", dayfirst=True)


summary = pd.read_csv(SUMMARY_PATH)
for column in ["selected_overlap_start", "selected_overlap_end", "first_observed_date", "monitoring_start"]:
    if column in summary.columns:
        summary[column] = parse_mixed_datetime(summary[column])

summary["site_id"] = summary["site_id"].astype("int64")
summary["selection_rank"] = summary["selection_rank"].astype("int64")

signal_profiles = sorted(summary["formula_profile"].dropna().unique().tolist())
if signal_profiles != [EXPECTED_SIGNAL_PROFILE]:
    raise ValueError(f"Expected only {EXPECTED_SIGNAL_PROFILE}, found {signal_profiles}")

nonnegative_summary = (
    summary.loc[summary["meaningful_negative_underlying_load_count_post_fill"].eq(0)]
    .sort_values("selection_rank")
    .reset_index(drop=True)
)
nonnegative_site_ids = [int(site_id) for site_id in nonnegative_summary["site_id"]]

if len(nonnegative_summary) != EXPECTED_NON_NEGATIVE_HOUSEHOLDS:
    raise AssertionError(
        f"Expected {EXPECTED_NON_NEGATIVE_HOUSEHOLDS} non-negative households, found {len(nonnegative_summary)}."
    )

site_dataset = ds.dataset(str(SITE_TIMESERIES_PATH), format="parquet")
site_table = site_dataset.to_table(
    columns=["site_id", "datetime", *TARGET_COLUMNS],
    filter=ds.field("site_id").isin(nonnegative_site_ids),
)
nonnegative_ts = site_table.to_pandas().sort_values(["site_id", "datetime"]).reset_index(drop=True)
nonnegative_ts["site_id"] = nonnegative_ts["site_id"].astype("int64")
nonnegative_ts["datetime"] = pd.to_datetime(nonnegative_ts["datetime"])

print(f"Non-negative households: {len(nonnegative_summary):,}")
print(f"5-minute rows loaded: {len(nonnegative_ts):,}")
print(f"Date range: {nonnegative_ts['datetime'].min()} to {nonnegative_ts['datetime'].max()}")
display(nonnegative_summary[[
    "selection_rank",
    "site_id",
    "state",
    "postcode",
    "dc_capacity_kw",
    "ac_capacity_kw",
    "min_underlying_load_kW_post_fill",
]].head(12))

<local path redacted>
  return pd.to_datetime(series, errors="coerce", dayfirst=True)
<local path redacted>
  return pd.to_datetime(series, errors="coerce", dayfirst=True)
<local path redacted>
  return pd.to_datetime(series, errors="coerce", dayfirst=True)


Non-negative households: 44
5-minute rows loaded: 4,625,280
Date range: 2024-02-26 00:00:00 to 2025-02-24 23:55:00


,selection_rank,site_id,state,postcode,dc_capacity_kw,ac_capacity_kw,min_underlying_load_kW_post_fill
0,1,1995273389,SA,5090,5.04,5.00,0.007010
1,2,245185730,VIC,3453,11.84,8.20,0.078340
2,3,905921552,NSW,2167,13.26,10.00,0.148773
3,4,1805446999,VIC,3040,13.30,10.00,0.168707
4,5,1284666164,VIC,3072,5.12,6.00,0.209110
5,6,1130090932,NSW,2758,5.13,5.00,0.033337
6,7,1429376445,NSW,2485,13.07,10.00,0.267180
7,8,1113741916,TAS,7325,10.40,9.99,-0.009843
8,9,2140545194,SA,5116,4.94,5.20,0.020017
9,10,1200712260,SA,5153,6.13,6.00,0.019017


## 4. Reconstruct PV And Battery Components

The selected-site parquet stores the three target signals. The next cell reconstructs PV generation and battery storage from those signals, then checks that the component identities hold before clustering.

In [4]:
# Purpose: reconstruct PV and battery components and validate the arithmetic identities.
# Positive battery_storage_kW means charging under the current selected signal convention.

nonnegative_ts = nonnegative_ts.copy()
nonnegative_ts["pv_generation_kW"] = nonnegative_ts["underlying_load_kW"] - nonnegative_ts["net_load_with_pv_kW"]
nonnegative_ts["battery_storage_kW"] = (
    nonnegative_ts["net_load_with_pv_and_battery_kW"] - nonnegative_ts["net_load_with_pv_kW"]
)
nonnegative_ts["battery_storage_abs_kW"] = nonnegative_ts["battery_storage_kW"].abs()
nonnegative_ts["is_negative_underlying"] = nonnegative_ts["underlying_load_kW"].lt(NEGATIVE_THRESHOLD_KW)

nonnegative_ts["weekday"] = nonnegative_ts["datetime"].dt.dayofweek
nonnegative_ts["hour"] = nonnegative_ts["datetime"].dt.hour
nonnegative_ts["minute"] = nonnegative_ts["datetime"].dt.minute
nonnegative_ts["week_slot_30min"] = (
    nonnegative_ts["weekday"] * PROFILE_SLOTS_PER_DAY
    + nonnegative_ts["hour"] * 2
    + nonnegative_ts["minute"] // 30
)

pv_identity_error = (
    nonnegative_ts["underlying_load_kW"]
    - nonnegative_ts["pv_generation_kW"]
    - nonnegative_ts["net_load_with_pv_kW"]
).abs().max()
battery_identity_error = (
    nonnegative_ts["net_load_with_pv_and_battery_kW"]
    - nonnegative_ts["battery_storage_kW"]
    - nonnegative_ts["net_load_with_pv_kW"]
).abs().max()
negative_rows = int(nonnegative_ts["is_negative_underlying"].sum())

if negative_rows != 0:
    raise AssertionError(f"The non-negative cohort contains {negative_rows:,} rows below {NEGATIVE_THRESHOLD_KW} kW.")
if pv_identity_error > 1e-9 or battery_identity_error > 1e-9:
    raise AssertionError(
        "Component identity check failed: "
        f"pv_error={pv_identity_error:.3e}, battery_error={battery_identity_error:.3e}"
    )

print(f"Rows below negative threshold: {negative_rows:,}")
print(f"PV identity max absolute error: {pv_identity_error:.3e} kW")
print(f"Battery identity max absolute error: {battery_identity_error:.3e} kW")

Rows below negative threshold: 0
PV identity max absolute error: 0.000e+00 kW
Battery identity max absolute error: 1.421e-14 kW


## 5. Method 1: PV And Battery Size Clustering

Method 1 clusters households using installed PV size and an operational battery-size proxy. PV size is `dc_capacity_kw`; battery size is the 95th percentile of absolute `battery_storage_kW` over the selected year.

In [5]:
# Purpose: calculate PV/battery size features and run 3-cluster KMeans.
# Features are standardized before KMeans so PV and battery size have comparable influence.

battery_size = (
    nonnegative_ts.groupby("site_id", observed=True)["battery_storage_abs_kW"]
    .quantile(0.95)
    .rename("battery_storage_abs_p95_kW")
    .reset_index()
)

method1_features = (
    nonnegative_summary[["selection_rank", "site_id", "state", "postcode", "dc_capacity_kw", "ac_capacity_kw"]]
    .merge(battery_size, on="site_id", how="left")
    .sort_values("selection_rank")
    .reset_index(drop=True)
)

if method1_features[METHOD1_FEATURES].isna().any().any():
    raise ValueError("Method 1 feature table contains missing values.")

method1_scaler = StandardScaler()
method1_scaled = method1_scaler.fit_transform(method1_features[METHOD1_FEATURES])
method1_kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=KMEANS_N_INIT)
method1_features["method1_cluster_id"] = method1_kmeans.fit_predict(method1_scaled)

display(method1_features[METHOD1_FEATURES + ["site_id", "selection_rank"]].describe().round(3))
print("Method 1 cluster counts:")
display(method1_features["method1_cluster_id"].value_counts().sort_index().rename("households"))

,dc_capacity_kw,battery_storage_abs_p95_kW,site_id,selection_rank
count,44.000,44.000,4.400000e+01,44.000
mean,9.612,2.539,1.131359e+09,22.500
std,4.570,1.099,6.080378e+08,12.845
min,4.940,0.001,3.023605e+07,1.000
25%,6.600,2.044,8.793464e+08,11.750
50%,7.510,2.584,1.101604e+09,22.500
75%,11.840,3.235,1.586596e+09,33.250
max,25.000,4.740,2.140545e+09,44.000


Method 1 cluster counts:


method1_cluster_id
0    12
1    24
2     8
Name: households, dtype: int64

## 6. Method 1: Representative Households

This section labels the three size clusters against the requested archetypes and selects the household nearest each cluster centroid as the representative.

In [6]:
# Purpose: label Method 1 clusters and choose the closest-to-centroid representative.
# Target matching is deterministic: each KMeans centroid is assigned to one requested archetype.

method1_target_points = np.array([
    [1.0, 1.0],   # big PV + big battery
    [-1.0, 1.0],  # small PV + big battery
    [1.0, -1.0],  # big PV + small battery
])

best_cost = np.inf
best_cluster_for_target = None
for cluster_perm in itertools.permutations(range(N_CLUSTERS)):
    cost = sum(
        np.linalg.norm(method1_kmeans.cluster_centers_[cluster_id] - method1_target_points[target_idx])
        for target_idx, cluster_id in enumerate(cluster_perm)
    )
    if cost < best_cost:
        best_cost = cost
        best_cluster_for_target = cluster_perm

method1_cluster_labels = {
    int(cluster_id): METHOD1_TARGET_LABELS[target_idx]
    for target_idx, cluster_id in enumerate(best_cluster_for_target)
}
method1_features["method1_cluster"] = method1_features["method1_cluster_id"].map(method1_cluster_labels)

method1_distances = np.linalg.norm(
    method1_scaled - method1_kmeans.cluster_centers_[method1_features["method1_cluster_id"].to_numpy()],
    axis=1,
)
method1_features["distance_to_method1_centroid"] = method1_distances

method1_representatives = (
    method1_features.sort_values("distance_to_method1_centroid")
    .groupby("method1_cluster", as_index=False, observed=True)
    .first()
    .sort_values("method1_cluster")
    .reset_index(drop=True)
)
method1_features["is_method1_representative"] = method1_features["site_id"].isin(method1_representatives["site_id"])

display(method1_representatives[[
    "method1_cluster",
    "selection_rank",
    "site_id",
    "dc_capacity_kw",
    "battery_storage_abs_p95_kW",
    "distance_to_method1_centroid",
]].round(3))

if PLOTLY_AVAILABLE:
    fig = px.scatter(
        method1_features,
        x="dc_capacity_kw",
        y="battery_storage_abs_p95_kW",
        color="method1_cluster",
        color_discrete_map=CLUSTER_COLORS,
        hover_data=["selection_rank", "site_id", "state", "postcode", "ac_capacity_kw"],
        title="Method 1: PV size and operational battery-size clustering",
        labels={
            "dc_capacity_kw": "PV DC capacity (kW)",
            "battery_storage_abs_p95_kW": "Battery storage p95 absolute power (kW)",
            "method1_cluster": "Cluster",
        },
    )
    fig.add_trace(
        go.Scatter(
            x=method1_representatives["dc_capacity_kw"],
            y=method1_representatives["battery_storage_abs_p95_kW"],
            mode="markers+text",
            marker={"symbol": "star", "size": 18, "color": "black", "line": {"color": "white", "width": 1}},
            text=method1_representatives["site_id"].astype(str),
            textposition="top center",
            name="Representative households",
            hovertemplate="representative site=%{text}<extra></extra>",
        )
    )
    fig.update_layout(height=560)
    fig.show()
else:
    print("Install plotly to render the Method 1 scatterplot.")
    display(method1_features[[
        "method1_cluster",
        "selection_rank",
        "site_id",
        "dc_capacity_kw",
        "battery_storage_abs_p95_kW",
        "is_method1_representative",
    ]].sort_values(["method1_cluster", "selection_rank"]).round(3))

,method1_cluster,selection_rank,site_id,dc_capacity_kw,battery_storage_abs_p95_kW,distance_to_method1_centroid
0,big PV + big battery,30,935185406,16.59,3.599,0.240
1,big PV + small battery,41,182060662,6.60,0.713,0.037
2,small PV + big battery,32,1690066375,7.62,2.522,0.071


## 7. Method 2: Typical Weekly Net-Load Profile Clustering

Method 2 clusters each household's 30-minute typical weekly profile for `net_load_with_pv_and_battery_kW`. The profile features are globally standardized so both profile shape and magnitude contribute to the KMeans distance.

In [7]:
# Purpose: build 30-minute typical weekly profiles and run 3-cluster KMeans.
# Grouping by 30-minute week slots averages the whole selected year into 336 profile points per household.

weekly_profile = (
    nonnegative_ts.groupby(["site_id", "week_slot_30min"], observed=True)[PROFILE_SIGNAL]
    .mean()
    .reset_index()
)

profile_matrix = (
    weekly_profile.pivot(index="site_id", columns="week_slot_30min", values=PROFILE_SIGNAL)
    .reindex(index=nonnegative_site_ids, columns=range(PROFILE_SLOTS_PER_WEEK))
)

if profile_matrix.isna().any().any():
    missing_count = int(profile_matrix.isna().sum().sum())
    raise ValueError(f"Typical weekly profile matrix contains {missing_count:,} missing values.")

profile_values = profile_matrix.to_numpy(dtype=float)
profile_global_mean = float(profile_values.mean())
profile_global_std = float(profile_values.std(ddof=0))
if profile_global_std == 0:
    raise ValueError("Profile feature matrix has zero global standard deviation.")

# Global standardization preserves household magnitude differences while keeping values numerically stable.
profile_scaled = (profile_values - profile_global_mean) / profile_global_std

method2_kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=KMEANS_N_INIT)
method2_cluster_ids = method2_kmeans.fit_predict(profile_scaled)

method2_features = nonnegative_summary[["selection_rank", "site_id", "state", "postcode", "dc_capacity_kw", "ac_capacity_kw"]].copy()
method2_features["method2_cluster_id"] = method2_cluster_ids

cluster_mean_load = (
    pd.DataFrame({
        "method2_cluster_id": range(N_CLUSTERS),
        "cluster_mean_profile_kW": method2_kmeans.cluster_centers_.mean(axis=1) * profile_global_std + profile_global_mean,
    })
    .sort_values("cluster_mean_profile_kW")
    .reset_index(drop=True)
)
method2_label_map = {
    int(row.method2_cluster_id): f"profile cluster {idx + 1}"
    for idx, row in cluster_mean_load.iterrows()
}
method2_features["method2_cluster"] = method2_features["method2_cluster_id"].map(method2_label_map)

profile_distances = np.linalg.norm(
    profile_scaled - method2_kmeans.cluster_centers_[method2_cluster_ids],
    axis=1,
)
method2_features["distance_to_method2_centroid"] = profile_distances

print(f"Typical weekly profile matrix: {profile_matrix.shape[0]} households x {profile_matrix.shape[1]} slots")
print("Method 2 cluster counts:")
display(method2_features["method2_cluster"].value_counts().sort_index().rename("households"))

Typical weekly profile matrix: 44 households x 336 slots
Method 2 cluster counts:


method2_cluster
profile cluster 1    11
profile cluster 2    29
profile cluster 3     4
Name: households, dtype: int64

## 8. Method 2: Representative Households

This section selects the household closest to each profile-cluster centroid, then visualizes every household profile with the representatives drawn more strongly.

In [8]:
# Purpose: choose Method 2 representatives and prepare long-form profile data for plotting.
# Representatives are closest to their cluster centroid in the globally standardized profile space.

method2_representatives = (
    method2_features.sort_values("distance_to_method2_centroid")
    .groupby("method2_cluster", as_index=False, observed=True)
    .first()
    .sort_values("method2_cluster")
    .reset_index(drop=True)
)
method2_features["is_method2_representative"] = method2_features["site_id"].isin(method2_representatives["site_id"])

profile_plot = (
    profile_matrix.reset_index()
    .melt(id_vars="site_id", var_name="week_slot_30min", value_name=PROFILE_SIGNAL)
    .merge(method2_features[["site_id", "selection_rank", "method2_cluster", "is_method2_representative"]], on="site_id", how="left")
)
profile_plot["week_slot_30min"] = profile_plot["week_slot_30min"].astype(int)

display(method2_representatives[[
    "method2_cluster",
    "selection_rank",
    "site_id",
    "dc_capacity_kw",
    "ac_capacity_kw",
    "distance_to_method2_centroid",
]].round(3))

if PLOTLY_AVAILABLE:
    fig = go.Figure()
    for row in method2_features.sort_values(["method2_cluster", "selection_rank"]).itertuples(index=False):
        site_profile = profile_plot.loc[profile_plot["site_id"].eq(row.site_id)]
        fig.add_trace(
            go.Scatter(
                x=site_profile["week_slot_30min"],
                y=site_profile[PROFILE_SIGNAL],
                mode="lines",
                name=f"site {int(row.site_id)}",
                legendgroup=row.method2_cluster,
                showlegend=False,
                line={
                    "color": CLUSTER_COLORS.get(row.method2_cluster),
                    "width": 1.1,
                },
                opacity=0.22,
                hovertemplate=f"site {int(row.site_id)}<br>slot=%{{x}}<br>net load=%{{y:.3f}} kW<extra></extra>",
            )
        )

    for row in method2_representatives.itertuples(index=False):
        site_profile = profile_plot.loc[profile_plot["site_id"].eq(row.site_id)]
        fig.add_trace(
            go.Scatter(
                x=site_profile["week_slot_30min"],
                y=site_profile[PROFILE_SIGNAL],
                mode="lines",
                name=f"{row.method2_cluster} representative: site {int(row.site_id)}",
                line={
                    "color": CLUSTER_COLORS.get(row.method2_cluster),
                    "width": 4.0,
                },
                hovertemplate=f"representative site {int(row.site_id)}<br>slot=%{{x}}<br>net load=%{{y:.3f}} kW<extra></extra>",
            )
        )

    for day in range(8):
        fig.add_vline(x=day * PROFILE_SLOTS_PER_DAY, line_width=0.7, line_color="rgba(120,120,120,0.35)")
    fig.add_hline(y=0, line_width=0.8, line_color="rgba(60,60,60,0.65)")
    fig.update_layout(
        title="Method 2: typical weekly net-load profiles by cluster",
        height=620,
        xaxis_title="Typical week",
        yaxis_title=f"{PROFILE_SIGNAL_LABEL} (kW)",
        legend_title_text="Representative households",
    )
    fig.update_xaxes(tickmode="array", tickvals=PROFILE_TICK_VALUES, ticktext=PROFILE_TICK_LABELS)
    fig.show()
else:
    print("Install plotly to render the Method 2 weekly profile plot.")
    display(method2_features[[
        "method2_cluster",
        "selection_rank",
        "site_id",
        "is_method2_representative",
        "distance_to_method2_centroid",
    ]].sort_values(["method2_cluster", "selection_rank"]).round(3))

,method2_cluster,selection_rank,site_id,dc_capacity_kw,ac_capacity_kw,distance_to_method2_centroid
0,profile cluster 1,42,1839862920,11.24,10.62,4.832
1,profile cluster 2,11,907278833,6.63,4.99,3.254
2,profile cluster 3,28,2026169230,13.30,10.00,14.878


## 9. Side-By-Side Cluster Interpretation

This section puts the two clustering methods next to each other. It is meant to make it easy to see whether a household's physical-size cluster agrees with its net-load-profile cluster.

In [9]:
# Purpose: compare Method 1 and Method 2 labels for the same 44 households.
# This is interpretive only; neither method is treated as ground truth.

combined_clusters = (
    method1_features[[
        "selection_rank",
        "site_id",
        "dc_capacity_kw",
        "battery_storage_abs_p95_kW",
        "method1_cluster",
        "is_method1_representative",
    ]]
    .merge(
        method2_features[[
            "site_id",
            "method2_cluster",
            "is_method2_representative",
            "distance_to_method2_centroid",
        ]],
        on="site_id",
        how="left",
    )
    .sort_values("selection_rank")
    .reset_index(drop=True)
)

cross_tab = pd.crosstab(combined_clusters["method1_cluster"], combined_clusters["method2_cluster"])

print("Cross-tab of physical-size clusters vs profile clusters:")
display(cross_tab)

print("Representatives selected by either method:")
display(
    combined_clusters.loc[
        combined_clusters["is_method1_representative"] | combined_clusters["is_method2_representative"]
    ].round(3)
)

Cross-tab of physical-size clusters vs profile clusters:


method2_cluster,profile cluster 1,profile cluster 2,profile cluster 3
method1_cluster,,,
big PV + big battery,5,3,4
big PV + small battery,3,5,0
small PV + big battery,3,21,0


Representatives selected by either method:


,selection_rank,site_id,dc_capacity_kw,battery_storage_abs_p95_kW,method1_cluster,is_method1_representative,method2_cluster,is_method2_representative,distance_to_method2_centroid
10,11,907278833,6.63,2.648,small PV + big battery,False,profile cluster 2,True,3.254
27,28,2026169230,13.30,3.240,big PV + big battery,False,profile cluster 3,True,14.878
29,30,935185406,16.59,3.599,big PV + big battery,True,profile cluster 1,False,19.109
31,32,1690066375,7.62,2.522,small PV + big battery,True,profile cluster 2,False,3.415
40,41,182060662,6.60,0.713,big PV + small battery,True,profile cluster 2,False,12.821
41,42,1839862920,11.24,2.397,small PV + big battery,False,profile cluster 1,True,4.832


## 10. Notes And Caveats

This section records the main limitations to keep in mind when interpreting the clusters.

In [10]:
# Purpose: print a compact reminder of interpretation caveats.
# These notes are deliberately kept in the notebook so the cluster plots are not over-interpreted.

print("Caveats:")
print("1. Battery size is an operational proxy: p95(abs(battery_storage_kW)), not a metadata capacity field.")
print("2. Method 1 uses physical/operational size only; it does not use the net-load shape.")
print("3. Method 2 uses a typical weekly profile of net_load_with_pv_and_battery_kW and preserves magnitude.")
print("4. KMeans gives simple exploratory groupings. Cluster labels and representatives are aids for inspection, not definitive household classes.")
print("5. This notebook writes no CSV, parquet, or figure files.")

Caveats:
1. Battery size is an operational proxy: p95(abs(battery_storage_kW)), not a metadata capacity field.
2. Method 1 uses physical/operational size only; it does not use the net-load shape.
3. Method 2 uses a typical weekly profile of net_load_with_pv_and_battery_kW and preserves magnitude.
4. KMeans gives simple exploratory groupings. Cluster labels and representatives are aids for inspection, not definitive household classes.
5. This notebook writes no CSV, parquet, or figure files.
